# RTI Governance Prototype v2: delay-based indicators (Source B)
**What changed from v1:** v1 used only outcome-label items (low discrimination, SE ≈ 0.97, so no usable latent trait). v2 uses **`cic_combined.csv`**, which has date columns, so we can build **delay (Responsiveness) indicators** that do *not* come from the outcome label.

**Trade-off:** Source B has only ~298 rows and many missing dates, so the usable sample will be small. Step 2 measures this *before* modelling. Sources A and B have no common key, so they **cannot be merged**; treat v2 as a separate analysis.

## Step 0: Install

In [ ]:
!pip -q install huggingface_hub girth pandas matplotlib numpy scipy

## Step 1: Load Source B and parse dates (format dd.mm.yyyy)

In [ ]:
import numpy as np, pandas as pd, re
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

path_b = hf_hub_download("joyboseroy/rti-bench", "cic_combined.csv", repo_type="dataset")
df = pd.read_csv(path_b)

DATE_COLS = ["rti_filed_on", "cpio_replied_on", "first_appeal_on", "faa_order_on",
             "second_appeal_on", "date_of_hearing", "date_of_decision", "date_signed"]
for c in DATE_COLS:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], format="%d.%m.%Y", errors="coerce")

print("Shape:", df.shape)

## Step 2: Coverage check (read this carefully)
How many rows actually have the fields we need? **This decides what is possible.**

In [ ]:
need = ["outcome", "exemptions_cited", "penalty_inr", "public_authority",
        "rti_filed_on", "cpio_replied_on", "first_appeal_on", "faa_order_on"]
need = [c for c in need if c in df.columns]
cov = pd.DataFrame({"non_null": df[need].notna().sum(), "pct": (df[need].notna().mean()*100).round(1)})
display(cov)

print("\nOutcome counts:\n", df["outcome"].value_counts(dropna=False).to_string())
print("\nRows with BOTH rti_filed_on and cpio_replied_on :", df[["rti_filed_on","cpio_replied_on"]].notna().all(axis=1).sum())
print("Rows with BOTH first_appeal_on and faa_order_on :", df[["first_appeal_on","faa_order_on"]].notna().all(axis=1).sum())
print("Rows with filed date but NO PIO reply date        :", (df.rti_filed_on.notna() & df.cpio_replied_on.isna()).sum())

## Step 3: Build indicators (0 = worse, higher = better)
| Indicator | Coding | Basis |
|---|---|---|
| `pio_speed` | 0 = >30 days, 1 = 16-30, 2 = ≤15 | RTI Act s.7(1): PIO must reply within 30 days |
| `faa_speed` | 0 = >45 days, 1 = 31-45, 2 = ≤30 | s.19(6): first appeal decided within 30 days, extendable to 45 |
| `compliance` | 0 = info directed / penalty, 1 = partial / remanded, 2 = appeal dismissed | outcome label (same assumption as v1) |
| `exemption_use` | 0 = 2+ exemptions, 1 = one, 2 = none | exemptions_cited |
| `no_penalty` | 0 = penalty, 1 = none | penalty_inr / outcome |

**Important:** a *missing* PIO reply date can mean "no reply happened" (a real transparency signal) **or** "the extractor missed it". The data cannot tell us which, so rows with a missing date are excluded from the delay items, not treated as late. State this in the report.

In [ ]:
def count_exemptions(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return 0
    s = str(v).strip()
    if s in ("", "[]", "None", "nan", "{}"): return 0
    hits = re.findall(r"8\s*\(\s*1\s*\)\s*\(\s*[a-j]\s*\)", s, flags=re.I)
    return len(hits) if hits else len([x for x in re.split(r"[,;|]", s) if x.strip()])

def bin_delay(days, cuts, labels):
    # days: Series; cuts: ascending upper bounds for labels[:-1]; invalid (<0 or >1000) -> NaN
    d = days.where((days >= 0) & (days <= 1000))
    out = pd.Series(np.nan, index=d.index)
    out[d.notna()] = labels[-1]
    for cut, lab in sorted(zip(cuts, labels[:-1]), reverse=True):
        out[d.notna() & (d > cut)] = lab
    return out

ind = pd.DataFrame(index=df.index)
oc = df["outcome"].astype(str).str.strip().str.upper().str.replace(" ", "_")
ind["outcome"] = oc
ind["authority"] = df.get("public_authority")

pio_days = (df["cpio_replied_on"] - df["rti_filed_on"]).dt.days
faa_days = (df["faa_order_on"] - df["first_appeal_on"]).dt.days
ind["pio_days"], ind["faa_days"] = pio_days, faa_days

# 0 if >30, 1 if 16-30, 2 if <=15   |   0 if >45, 1 if 31-45, 2 if <=30
def pio_code(d):
    if pd.isna(d) or d < 0 or d > 1000: return np.nan
    return 0 if d > 30 else (1 if d > 15 else 2)
def faa_code(d):
    if pd.isna(d) or d < 0 or d > 1000: return np.nan
    return 0 if d > 45 else (1 if d > 30 else 2)
ind["pio_speed"] = pio_days.map(pio_code)
ind["faa_speed"] = faa_days.map(faa_code)

comp_map = {"INFORMATION_DIRECTED": 0, "PENALTY_IMPOSED": 0, "PARTIAL_RELIEF": 1, "REMANDED": 1, "APPEAL_DISMISSED": 2}
ind["compliance"] = oc.map(comp_map)

n_ex = df["exemptions_cited"].map(count_exemptions) if "exemptions_cited" in df else pd.Series(0, index=df.index)
ind["exemption_use"] = n_ex.map(lambda n: 2 if n == 0 else (1 if n == 1 else 0))

pen = (df["penalty_inr"].fillna(0) > 0).astype(int) if "penalty_inr" in df else 0
pen = pen | (oc == "PENALTY_IMPOSED").astype(int)
ind["no_penalty"] = 1 - pen

CANDIDATES = ["pio_speed", "faa_speed", "compliance", "exemption_use", "no_penalty"]
print("Non-missing count per candidate item:")
print(ind[CANDIDATES].notna().sum().to_string())
for c in CANDIDATES:
    print(f"\n{c}:\n", ind[c].value_counts().sort_index().to_string())

## Step 4: Choose items and sample (automatic screening, then YOU decide)
Rules: drop items with >90% in one category, and items with fewer than `MIN_ITEM_N` non-missing values. Then keep only rows complete on the remaining items (complete-case analysis).
Edit `MIN_ITEM_N` or `FORCE_DROP` if needed.

In [ ]:
MIN_ITEM_N = 80
FORCE_DROP = []          # e.g. ["no_penalty"]

ITEMS = []
for c in CANDIDATES:
    s = ind[c].dropna()
    if c in FORCE_DROP:
        print(f"drop {c}: forced"); continue
    if len(s) < MIN_ITEM_N:
        print(f"drop {c}: only {len(s)} non-missing"); continue
    if s.value_counts(normalize=True).max() > 0.90:
        print(f"drop {c}: >90% one category"); continue
    ITEMS.append(c)

use = ind.dropna(subset=ITEMS).copy()
use[ITEMS] = use[ITEMS].astype(int)
print("\nItems used:", ITEMS)
print("Complete cases:", len(use), "of", len(df))
if len(use) < 100:
    print("WARNING: n < 100. Parameter estimates will be unstable; treat this as a demonstration only.")
if len(ITEMS) < 3:
    print("WARNING: fewer than 3 items. GRM is weakly identified.")

print("\nSpearman correlations:\n", use[ITEMS].corr(method="spearman").round(2))

## Step 5: Fit GRM + diagnostics
Rule-of-thumb flags: discrimination < 0.65 = low, thresholds beyond ±4 = unstable.

In [ ]:
from girth import grm_mml
D = use[ITEMS].values.T.astype(int)
res = grm_mml(D)
a = np.asarray(res["Discrimination"]); b = np.asarray(res["Difficulty"])

params = pd.DataFrame({"item": ITEMS, "discrimination": a.round(2)})
for k in range(b.shape[1]):
    params[f"threshold_{k+1}"] = b[:, k].round(2)
display(params)

for i, it in enumerate(ITEMS):
    if a[i] < 0.65: print(f"LOW discrimination: {it} ({a[i]:.2f})")
    if np.nanmax(np.abs(b[i])) > 4: print(f"UNSTABLE thresholds: {it}")

## Step 6: Latent scores + reliability

In [ ]:
def grm_eap(D, a, b, n_quad=61):
    th = np.linspace(-5, 5, n_quad)
    prior = np.exp(-0.5 * th**2); prior /= prior.sum()
    n_items, n_people = D.shape
    loglik = np.zeros((n_people, n_quad))
    for i in range(n_items):
        bi = b[i][~np.isnan(b[i])]
        cum = 1 / (1 + np.exp(-a[i] * (th[None, :] - bi[:, None])))
        cum = np.vstack([np.ones(n_quad), cum, np.zeros(n_quad)])
        probs = np.clip(cum[:-1] - cum[1:], 1e-12, 1)
        loglik += np.log(probs[D[i].astype(int)])
    post = np.exp(loglik - loglik.max(1, keepdims=True)) * prior
    post /= post.sum(1, keepdims=True)
    eap = post @ th
    sd = np.sqrt(np.maximum(post @ th**2 - eap**2, 0))
    return eap, sd

use["theta"], use["theta_se"] = grm_eap(D, a, b)
rel = 1 - (use["theta_se"]**2).mean()      # marginal reliability, prior variance = 1
print(use[["theta", "theta_se"]].describe().round(2))
print(f"\nMarginal reliability ~ {rel:.2f}   (rule of thumb: >= 0.70 acceptable, v1 was ~0.06)")

## Step 7: Charts
Note: `compliance` is built from `outcome`, so score-by-outcome is partly circular. Prefer the **delay vs score** view for interpretation.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(use["theta"], bins=25); ax[0].set_title("Latent score distribution"); ax[0].set_xlabel("theta")
ax[1].bar(ITEMS, a); ax[1].axhline(0.65, ls="--"); ax[1].set_title("Item discrimination (dashed = 0.65)")
ax[1].tick_params(axis="x", rotation=30)
if "pio_days" in use:
    m = use["pio_days"].between(0, 200)
    ax[2].scatter(use.loc[m, "pio_days"], use.loc[m, "theta"], alpha=0.5)
    ax[2].set_title("PIO delay (days) vs theta"); ax[2].set_xlabel("days")
plt.tight_layout(); plt.show()

## What to report (be upfront)
1. **Sample:** state the complete-case n, and how many rows were lost to missing dates.
2. **Missingness:** missing reply date is ambiguous (no reply vs extraction gap). Excluded, not treated as late.
3. **Reliability and discrimination:** report them even if weak. "Items from RTI-Bench alone support / do not support a stable latent trait" is a valid finding.
4. **Selection bias:** only disputed CIC cases.
5. **Coding assumptions:** day thresholds follow RTI Act s.7(1) and s.19(6); the outcome coding is a modelling choice.
6. **Next step if still weak:** more items (second-appeal delay, hearing gaps), a bigger dated dataset (e.g. the Kotoky-Saradhi query log), or compare with CFA.